# 01 - Zero-shot Baseline (Qwen3.5-4B-Base)

Baseline MAE on 200 test items WITHOUT fine-tuning. Run before training to establish comparison.

ETA: ~10 minutes on RTX 5090 32GB.

In [ ]:
# Cell 1 - Imports + constants
import os, sys, json, torch, random
sys.path.insert(0, os.path.abspath("."))

from transformers import AutoTokenizer, AutoModelForCausalLM
from utils.items_vn import load_items
from utils.evaluator_vn import VnTester, _rmsle
from utils.training_utils import get_bnb_config, MAX_NEW_TOKENS

BASE_MODEL   = "Qwen/Qwen3.5-4B-Base"
RESULTS_PATH = "results/zero_shot_results.json"
EVAL_SIZE    = 200

In [ ]:
# Cell 2 - Load base model (no adapter). torch_dtype=bfloat16 BAT BUOC.
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=get_bnb_config(),
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()
print(f"VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# Cell 3 - Test items (seed=42 reproducible)
random.seed(42)
all_test = load_items("test")
test_items = random.sample(all_test, min(EVAL_SIZE, len(all_test)))
print(f"Test items: {len(test_items)}")
print(f"Price range: {min(i.price for i in test_items):.0f}K - {max(i.price for i in test_items):.0f}K VND")

In [ ]:
# Cell 4 - Predictor + evaluate
import re
def zero_shot_predict(item) -> float:
    inputs = tokenizer(item.prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    prompt_len = inputs["input_ids"].shape[1]
    text = tokenizer.decode(out[0, prompt_len:], skip_special_tokens=True).strip()
    m = re.search(r"\d+\.?\d*", text)
    return float(m.group()) if m else 0.0

tester = VnTester(zero_shot_predict, test_items,
                  title="Qwen3.5-4B Zero-shot", size=EVAL_SIZE, workers=1)
tester.run()

In [ ]:
# Cell 5 - Save results JSON
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score

results = {
    "model": BASE_MODEL,
    "mode": "zero_shot",
    "eval_size": EVAL_SIZE,
    "mae_k_vnd": round(float(np.mean(tester.errors)), 2),
    "rmsle": round(_rmsle(tester.truths, tester.guesses), 4),
    "mse": round(float(mean_squared_error(tester.truths, tester.guesses)), 2),
    "r2": round(float(r2_score(tester.truths, tester.guesses)), 4),
}
os.makedirs("results", exist_ok=True)
with open(RESULTS_PATH, "w") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print(json.dumps(results, indent=2))